# 01 — QAOA pour MaxCut, et comment l'IA l'aide à démarrer

1. Le problème MaxCut
2. Le circuit QAOA avec PennyLane
3. Le paysage d'énergie : à quoi ressemble ce qu'on optimise
4. Optimiser, puis lire la solution
5. Les angles optimaux se ressemblent d'un graphe à l'autre…
6. … donc un modèle d'apprentissage peut les prédire

> Noyau à sélectionner : **Python (qaoa-ml)** (créé par `setup.bat`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import pennylane as qml
from qaoaml import *

rng = np.random.default_rng(4)

## 1. MaxCut

On veut répartir les sommets d'un graphe en deux groupes (0 et 1) pour **couper le plus d'arêtes possible**
(une arête est coupée si ses deux extrémités sont dans des groupes différents).
Il y a $2^n$ partitions : facile pour 8 sommets, impossible à énumérer pour 100.
Le problème est NP-difficile ; on s'en sert comme banc d'essai des algorithmes d'optimisation quantique.

In [ ]:
g = random_graph(rng, 8, 8)
c_max, best = brute_force(g)
print(f"{g.number_of_nodes()} sommets, {g.number_of_edges()} arêtes — coupe maximale : {c_max}")

pos = nx.spring_layout(g, seed=1)
cut_edges = [(i, j) for i, j in g.edges() if best[i] != best[j]]
nx.draw(g, pos, node_color=["#6a4c93" if b else "#c9ced6" for b in best], with_labels=True,
        edgelist=cut_edges, edge_color="#d1495b", width=2)
nx.draw_networkx_edges(g, pos, edgelist=[e for e in g.edges() if e not in cut_edges], style="dashed")
plt.title("Solution optimale (arêtes coupées en rouge)"); plt.show()

## 2. Le circuit QAOA

On encode chaque partition dans un état de base $|z\rangle$ de $n$ qubits. QAOA prépare
$$|\gamma,\beta\rangle = U_M(\beta_p)U_C(\gamma_p)\cdots U_M(\beta_1)U_C(\gamma_1)\,|+\rangle^{\otimes n}$$
- $U_C(\gamma)$ ajoute à chaque partition une **phase proportionnelle à sa coupe** ;
- $U_M(\beta)$ (des rotations $R_X$) **mélange** les partitions pour que ces phases interfèrent.

Bien réglées, les interférences amplifient les bonnes coupes. Seuls $2p$ angles sont à régler.

In [ ]:
circuit, cost_h = pennylane_circuit(g, p=1)
print(qml.draw(circuit, level="device", decimals=2)(np.array([0.5, -0.3]))[:600], "…")

## 3. Le paysage d'énergie ($p = 1$)

Avec $p=1$ il n'y a que deux angles : on peut **tout dessiner**. Chaque pixel est une exécution du circuit.
On utilise le simulateur rapide du projet (identique à PennyLane, ~50 fois plus rapide).

In [ ]:
sim = QAOASimulator(g)
print("Vérification simulateur vs PennyLane :", sim.expected_cut([0.5, -0.3]), -float(circuit(np.array([0.5, -0.3]))))

G, B = np.meshgrid(np.linspace(-np.pi / 2, np.pi / 2, 80), np.linspace(-np.pi / 4, np.pi / 4, 80))
Z = np.vectorize(lambda a, b: sim.approx_ratio([a, b]))(G, B)
plt.figure(figsize=(6, 4))
plt.contourf(G, B, Z, levels=30, cmap="viridis"); plt.colorbar(label="ratio d'approximation")
plt.xlabel(r"$\gamma$"); plt.ylabel(r"$\beta$"); plt.title("Paysage QAOA, p = 1"); plt.show()

Le paysage a plusieurs bosses : selon le point de départ, l'optimiseur peut finir dans un **mauvais maximum local**.
C'est pour cela que l'initialisation compte.

## 4. Optimiser et lire la solution

In [ ]:
for p in (1, 2, 3):
    res = optimize(sim, ramp_init(p))
    print(f"p = {p} : ratio = {res.approx_ratio:.3f} après {res.nfev} évaluations du circuit")

top = sim.most_likely(res.params, k=5)
print("\nPartitions les plus probables (p = 3) :")
for bits, prob, cut in top:
    print(f"  {''.join(map(str, bits))}  probabilité {prob:.3f}  coupe {cut}/{c_max}")

Le ratio moyen n'est pas 1, mais en **mesurant** quelques fois le circuit on tombe vite sur la coupe optimale.
À comparer : la recherche locale classique trouve l'optimum en une fraction de milliseconde sur ces petits graphes.

## 5. Concentration des paramètres

Optimisons QAOA ($p = 2$) sur 40 graphes différents et regardons les angles obtenus.

In [ ]:
df = build_dataset(n_graphs=40, p=2, seed=0)
fig, ax = plt.subplots(figsize=(6, 3.5))
for k, lab in enumerate([r"$\gamma_1$", r"$\gamma_2$", r"$\beta_1$", r"$\beta_2$"]):
    ax.scatter(np.full(len(df), k) + rng.normal(0, 0.05, len(df)), df[f"param_{k}"], s=12, alpha=0.6)
ax.set_xticks(range(4), [r"$\gamma_1$", r"$\gamma_2$", r"$\beta_1$", r"$\beta_2$"])
ax.set_ylabel("angle optimal (rad)"); ax.set_title("40 graphes, 40 optimisations"); plt.show()

Les angles optimaux sont **regroupés** : c'est la *concentration des paramètres*. Ils dépendent un peu
de la structure du graphe (degré, densité…) — exactement ce qu'un modèle d'apprentissage peut capter.

## 6. Prédire les angles avec un modèle d'apprentissage

In [ ]:
test = build_dataset(n_graphs=15, p=2, seed=1)
res = evaluate(df, test, strategies=["random", "ramp", "mean", "knn", "rf"])
summarize(res)

- **Ratio initial** : qualité des angles proposés *sans aucune optimisation*.
- **Évaluations** : appels au circuit avant convergence — sur un vrai ordinateur quantique, c'est ce qui coûte.

Pour l'étude complète (200 graphes, $p = 1$ à $3$, généralisation à des graphes plus grands) :
`python experiments/run_study.py`.

## Exercices
1. Dans la partie 3, dessinez le paysage d'un graphe **biparti** (`nx.cycle_graph(8)`). Que vaut le ratio maximal ?
2. Augmentez le nombre de graphes d'entraînement (partie 5). Le ratio initial du modèle `knn` progresse-t-il ?
3. Testez un graphe de 12 sommets avec un modèle entraîné sur 6 à 10 sommets.